In [1]:
import rasterio
import rasterio.mask
from rasterio.mask import mask as rio_mask
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

AUTOCORRELAÇÃO ESPACIAL DE AMOSTRAS

Funções

In [2]:
def autocorrelation(clipped_array, title, target_correlation = 0.4 , plot = False):
    
    shape = np.shape(clipped_array)
    n_col =  shape[1]
    n_row =  shape[0]
    data =  clipped_array
    
    # Convolving mask
    k1 = min(int(n_col / 3) + 2, 20) #mask extention Ydim 
    k2 = min(int(n_row / 3) + 2, 20) #mask extention Xdim


    # Statistic of valid pixels
    mean = np.mean(data[data >= 0])
    n_valid_pixels = np.count_nonzero(data[data >= 0])
    # 
    C = np.zeros((k1, k2))
    te = np.zeros((k1, k2), dtype=int)

    for i in range(1, k1 + 1):
        for j in range(1, k2 + 1):
            for k in range(n_col - i + 1):
                for l in range(n_row - j + 1):
                    lj = (l * n_col) + k
                    temp = lj + ((j - 1) * n_col) + i - 1
                    if data.flat[lj] >= 0 and data.flat[temp] >= 0:
                        C[i-1, j-1] += (data.flat[lj] - mean) * (data.flat[temp] - mean)
                        te[i-1, j-1] += 1

    # Normalização da correlação
    var = C[0, 0]/n_valid_pixels
                        
    norm_factor = (te * var) # normalization factor
    R = C/norm_factor     

    hh = np.concatenate([np.flip(R[:k1, 0]), R[1:k1, 0]])
    vv = np.concatenate([np.flip(R[0, :k2]), R[0, 1:k2]])
    
    #print(hh)
    #print(vv)
    min_lag_h, _ = find_nearest(hh,  target_correlation)
    min_lag_v, _ = find_nearest(vv,  target_correlation)

    
    if plot:
        
        plt.style.use('dark_background')
        lags_h = np.arange(1 - k1, k1)
        lags_v = np.arange(1 - k2, k2)

        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.plot(lags_h, hh,'x-', color='white')
        plt.axhline(y=target_correlation, color='r', linestyle='--')
        plt.axvline(x=min_lag_h, color='yellow', linestyle='--')
        plt.title(f'{title} - Horizontal')
        plt.xlabel("Lag")
        plt.ylabel("Autocorrelation")

        plt.subplot(1, 2, 2)
        plt.plot(lags_v, vv,'x-', color='white')
        plt.axhline(y=target_correlation, color='r', linestyle='--')
        plt.axvline(x=min_lag_v, color='yellow', linestyle='--')
        plt.title(f'{title} - Vertical')
        plt.xlabel("Lag")
        plt.ylabel("Autocorrelation")

        plt.tight_layout()
        plt.show()

    
    return(min_lag_h, min_lag_v)
   

In [3]:
def find_nearest(array, value):
    center = len(array)//2
    for i in range(center, len(array)):
        if array[i] < value:
            return i - center, array[i]  # Retorna o lag e o valor
    return None, None 

In [18]:
def extrair_dados_raster(polygon, image_array, transform):
    """
    Extrai os dados raster de um dado polígono.
    """
                
    if not polygon.is_valid:
        print("Polígono inválido encontrado.")
        return None

    try:
        # Calcular a janela para o recorte baseado no bounding box do polígono
        minx, miny, maxx, maxy = polygon.bounds
        window = from_bounds(minx, miny, maxx, maxy, transform)

        # Recortar o array com base na janela
        window_array = image_array[window.toslices()]**0.5

        return window_array
    except Exception as e:
        print(f"Erro ao processar o polígono: {e}")
        return None

In [5]:
def calculate_iqr(lags):
    q1 = np.percentile(lags, 25)
    q3 = np.percentile(lags, 75)
    iqr = q3 - q1
    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)
    
    # Identifica valores que são não outliers e estão dentro dos limites
    non_outliers = [x for x in lags if lower_bound <= x <= upper_bound]
    
    return non_outliers

def find_max_lag_without_outliers(lags, IQR = True):
    if IQR:
        non_outliers = calculate_iqr(lags)
        max_lag = max(non_outliers) if non_outliers else None
    else:
        max_lag = np.max(lags)
    return max_lag

In [6]:
def bounding_box(shapefile_path, output_path, nome = None):
    # Carregar o arquivo shapefile original
    gdf = gpd.read_file(shapefile_path)
    
    # Criar uma nova GeoDataFrame para os polígonos externos
    gdf_bbox = gdf.copy()
    
    # Calcular o menor polígono externo para cada polígono
    gdf_bbox['geometry'] = gdf.geometry.apply(lambda x: x.envelope)
    
    # Salvando o novo shapefile
    gdf_bbox.to_file(output_path+nome)


Lendos Paths para polígonos e imagens

In [7]:
raster_path = 'G:/Meu Drive/INPE/projeto_dissertacao/0_images/1_processed_images/0_SAR/2010/REC_2010_ALPSRP234707120_Cal_ML_Spk3_DSk_TC_modificado_aligned.TIF'
shapefile_path = 'G:/Meu Drive/INPE/projeto_dissertacao/samples_design/reis_vec_samples/Amostras/amostras_2010.shp'

In [8]:
#Importando os vetores
shapes = gpd.read_file(shapefile_path)
shapes.shape

(127, 4)

In [16]:
import math

In [19]:
# Agrupando os polígonos por classe
# OBS.:Shapefile DEVE conter uma coluna com o nome 'Class'
polygons_por_classe = shapes.groupby('Classe')
IQR_ = True
plot_ = False
# Dicionário para armazenar os lags horizontais e verticais por classe
# Lista para armazenar os dicionários por banda

lags_por_banda = []
with rasterio.open(raster_path) as src:
    
    transform = src.transform
    # Looping por banda 
    for banda in range(1,3):
        print(f'Processameto banda {banda}')
        lags_por_classe = {}
        
        # Looping por classe
        for classe, geoms in polygons_por_classe:
            lags_horizontais = []  # Lista para armazenar lags horizontais de cada polígono na classe atual
            lags_verticais = []  # Lista para armazenar lags verticais de cada polígono na classe atual
            print(f"Processando classe {classe}.")

            # Looping por amostra
            for _,row in geoms.iterrows():
                try: # Pode não encontrar um lag minínimo para uma correlação abaixo do limiar dado
                    masked_band, _ = rio_mask(src, [row['geometry']], crop=False, indexes=banda, invert=False, filled=True, nodata=-1)
                    # Atribui o valor de -1 para os pixels fora do recorte
                    masked_band[masked_band == src.nodata] = -1
                    clipped_array = extrair_dados_raster(row['geometry'], masked_band, transform)
                    if clipped_array is not None:
                        title = f" Amostra {row.name + 1} - Classe {classe} (Banda {banda})" #título do plot, caso ele seja TRUE
                        lag_horizontal, lag_vertical = autocorrelation(clipped_array,title, target_correlation = 0.3,  plot = plot_) 
                        if lag_horizontal is not None:
                            lags_horizontais.append(lag_horizontal)
                        if lag_vertical is not None:
                            lags_verticais.append(lag_vertical)
                except Exception as e:
                    print(f"Erro ao calcular lags do polígono {row.name  + 1}: {e}")
                    continue  # Pula para o próximo polígono dentro da mesma classe
            print(f"Lags Horizontais {lags_horizontais}")
            print(f"Lags Verticais {lags_verticais}")

            
        # Calculo dos lag medianos e máximos nas direções horizontal e vertical para a classe atual (se houver lags válidos)
            if lags_horizontais:
                lag_mediano_horizontal = np.median(lags_horizontais)
                lag_max_horizontal = find_max_lag_without_outliers(lags_horizontais, IQR = IQR_)
                print(f"Lag mediano horizontal para a classe {classe}: {lag_mediano_horizontal}")
                print(f"Lag máximo horizontal para a classe {classe}: {lag_max_horizontal}")
            else:
                print(f"Não foram encontrados lags horizontais válidos para a classe {classe}")
                lag_mediano_horizontal = None
                lag_max_horizontal = None

            if lags_verticais:
                lag_mediano_vertical = np.median(lags_verticais)
                lag_max_vertical = find_max_lag_without_outliers(lags_verticais, IQR = IQR_)
                print(f"Lag mediano vertical para a classe {classe}: {lag_mediano_vertical}")
                print(f"Lag máximo vertical para a classe {classe}: {lag_max_vertical}")
            else:
                print(f"Não foram encontrados lags verticais válidos para a classe {classe}")
                lag_mediano_vertical = None
                lag_max_vertical = None
            print('-----------------------------------------------------------------------------------')
            lags_por_classe[classe] = {
                'lags_horizontais': lags_horizontais,
                'lag_mediano_horizontal': lag_mediano_horizontal,
                'lags_verticais': lags_verticais,
                'lag_mediano_vertical': lag_mediano_vertical,
                'lag_max_horizontal': lag_max_horizontal,
                'lag_max_vertical': lag_max_vertical 
            }
        
        lags_por_banda.append(lags_por_classe)
        print('------------------------------------------------ ----------------------------------------')
        print(f'Fim do calculo do lag para banda {banda}')
        print('----------------------------------------------------------------------------------------')

Processameto banda 1
Processando classe AP.


C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.

Lags Horizontais [2, 3, 3, 1, 3, 2, 2, 7, 2, 2, 3, 2, 3, 4]
Lags Verticais [3, 3, 2, 2, 3, 2, 3, 3, 2, 2, 2, 8, 3, 5]
Lag mediano horizontal para a classe AP: 2.5
Lag máximo horizontal para a classe AP: 4
Lag mediano vertical para a classe AP: 3.0
Lag máximo vertical para a classe AP: 3
-----------------------------------------------------------------------------------
Processando classe CL.


C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\1808577864.py:34: RuntimeWarning: invalid value encountered in divide
  R = C/norm_factor
C:\Users\PC\AppData\Local\Te

Lags Horizontais [3, 3, 3, 3, 8, 3, 2, 4, 2, 2, 3, 3, 2, 3, 2, 3, 4]
Lags Verticais [2, 3, 3, 2, 7, 2, 2, 3, 3, 3, 3, 2, 2, 3, 4, 3, 3]
Lag mediano horizontal para a classe CL: 3.0
Lag máximo horizontal para a classe CL: 4
Lag mediano vertical para a classe CL: 3.0
Lag máximo vertical para a classe CL: 4
-----------------------------------------------------------------------------------
Processando classe FD.


...

Lags Horizontais [2, 2, 3, 3, 3, 3, 3, 2, 2, 2, 3]
Lags Verticais [2, 2, 2, 3, 2, 3, 2, 2, 7, 5, 2, 3]
Lag mediano horizontal para a classe SS1: 3.0
Lag máximo horizontal para a classe SS1: 3
Lag mediano vertical para a classe SS1: 2.0
Lag máximo vertical para a classe SS1: 3
-----------------------------------------------------------------------------------
Processando classe SS2.


C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.

Lags Horizontais [3, 2, 3, 2, 2, 2, 3, 2, 1, 3, 3]
Lags Verticais [3, 3, 2, 2, 2, 3, 3, 3, 3, 1, 3, 2]
Lag mediano horizontal para a classe SS2: 2.0
Lag máximo horizontal para a classe SS2: 3
Lag mediano vertical para a classe SS2: 3.0
Lag máximo vertical para a classe SS2: 3
-----------------------------------------------------------------------------------
Processando classe SS3.


C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.5
C:\Users\PC\AppData\Local\Temp\ipykernel_30796\591477439.py:16: RuntimeWarning: invalid value encountered in sqrt
  window_array = image_array[window.toslices()]**0.

Lags Horizontais [2, 3, 2, 3, 2, 2, 2, 2, 3]
Lags Verticais [3, 3, 2, 2, 3, 2, 3, 3, 2]
Lag mediano horizontal para a classe SS3: 2.0
Lag máximo horizontal para a classe SS3: 3
Lag mediano vertical para a classe SS3: 3.0
Lag máximo vertical para a classe SS3: 3
-----------------------------------------------------------------------------------
------------------------------------------------ ----------------------------------------
Fim do calculo do lag para banda 2
----------------------------------------------------------------------------------------


In [20]:
pd.DataFrame(lags_por_banda[0])

,AP,CL,FD,FP,PL,PS,SE,SS1,SS2,SS3
lags_horizontais,"[2, 3, 3, 1, 3, 2, 2, 7, 2, 2, 3, 2, 3, 4]","[3, 3, 3, 3, 8, 3, 2, 4, 2, 2, 3, 3, 2, 3, 2, ...","[2, 2, 2, 3, 2, 3, 2, 2]","[2, 2, 2, 2, 2, 2, 3, 3]","[3, 2, 3, 3, 1, 4, 2, 2, 6, 3, 3, 2, 3, 3, 2, 2]","[3, 3, 3, 2, 2, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3]","[4, 2, 3, 3, 5, 5, 4, 4, 3, 3, 3, 2, 3, 2, 3, 7]","[3, 2, 2, 3, 2, 3, 2, 2, 3, 3, 2, 2]","[2, 2, 3, 2, 3, 2, 2, 2, 2, 2, 2, 2]","[4, 3, 3, 2, 3, 3, 2, 3, 3]"
lag_mediano_horizontal,2.5,3.0,2.0,2.0,3.0,3.0,3.0,2.0,2.0,3.0
lags_verticais,"[3, 3, 2, 2, 3, 2, 3, 3, 2, 2, 2, 8, 3, 5]","[2, 3, 3, 2, 7, 2, 2, 3, 3, 3, 3, 2, 2, 3, 4, ...","[2, 2, 3, 2, 3, 2, 3, 2]","[2, 2, 2, 2, 2, 2, 3, 3]","[3, 2, 3, 3, 2, 3, 2, 3, 3, 3, 2, 2, 3, 2, 2, 2]","[3, 3, 2, 2, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2]","[3, 2, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3]","[3, 2, 2, 3, 3, 3, 2, 2, 5, 3, 2, 3]","[2, 3, 2, 3, 3, 2, 2, 3, 4, 2, 4, 2]","[2, 3, 3, 3, 2, 3, 2, 3, 3]"
lag_mediano_vertical,3.0,3.0,2.0,2.0,2.5,3.0,3.0,3.0,2.5,3.0
lag_max_horizontal,4,4,2,2,4,3,5,3,2,3
lag_max_vertical,3,4,3,2,3,3,3,3,4,3


In [21]:
pd.DataFrame(lags_por_banda[1])

,AP,CL,FD,FP,PL,PS,SE,SS1,SS2,SS3
lags_horizontais,"[3, 3, 3, 1, 2, 2, 2, 3, 3, 3, 3, 2, 2, 3]","[2, 3, 6, 2, 6, 2, 3, 3, 2, 2, 2, 3, 2, 3, 2, ...","[3, 2, 3, 2, 2, 2, 2, 2]","[2, 2, 2, 2, 2, 2, 3, 2]","[3, 3, 4, 3, 3, 8, 2, 2, 3, 3, 3, 3, 2, 2, 2, 3]","[3, 7, 2, 2, 2, 3, 4, 2, 3, 3, 3, 3, 3, 2, 3]","[3, 3, 3, 3, 7, 4, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3]","[2, 2, 3, 3, 3, 3, 3, 2, 2, 2, 3]","[3, 2, 3, 2, 2, 2, 3, 2, 1, 3, 3]","[2, 3, 2, 3, 2, 2, 2, 2, 3]"
lag_mediano_horizontal,3.0,3.0,2.0,2.0,3.0,3.0,3.0,3.0,2.0,2.0
lags_verticais,"[3, 3, 2, 3, 2, 2, 3, 2, 3, 2, 3, 3, 3, 3]","[3, 3, 3, 3, 7, 2, 2, 4, 2, 3, 3, 2, 3, 2, 4, ...","[3, 2, 3, 2, 2, 2, 2, 2]","[2, 2, 2, 2, 2, 2, 3, 2]","[3, 3, 2, 3, 3, 3, 2, 3, 2, 4, 3, 3, 3, 2, 2]","[3, 3, 2, 2, 3, 4, 2, 3, 3, 3, 3, 3, 2, 3]","[3, 2, 2, 2, 3, 2, 3, 3, 2, 2, 3, 2, 3, 3, 2, 2]","[2, 2, 2, 3, 2, 3, 2, 2, 7, 5, 2, 3]","[3, 3, 2, 2, 2, 3, 3, 3, 3, 1, 3, 2]","[3, 3, 2, 2, 3, 2, 3, 3, 2]"
lag_mediano_vertical,3.0,3.0,2.0,2.0,3.0,3.0,2.0,2.0,3.0,3.0
lag_max_horizontal,3,3,2,2,4,4,3,3,3,3
lag_max_vertical,3,4,2,2,4,4,3,3,3,3


In [22]:
# Dicionário para armazenar os maiores lags
maiores_lags = {}

# Iterando sobre cada dicionário da lista referente aos lags nas polarizações HV e HH
for dicionario in lags_por_banda:
    for classe, dados in dicionario.items():
        # Conferindo se a classe já foi verificada anteriormente
        if classe not in maiores_lags:
            maiores_lags[classe] = {'lag_max_horizontal': 0, 'lag_max_vertical': 0}
            
        # Atualiza os maiores lags ENTRE AS BANDAS
        maiores_lags[classe]['lag_max_horizontal'] = max(maiores_lags[classe]['lag_max_horizontal'], dados['lag_max_horizontal'])
        maiores_lags[classe]['lag_max_vertical'] = max(maiores_lags[classe]['lag_max_vertical'], dados['lag_max_vertical'])

print(pd.DataFrame(maiores_lags))

                    AP  CL  FD  FP  PL  PS  SE  SS1  SS2  SS3
lag_max_horizontal   4   4   2   2   4   4   5    3    3    3
lag_max_vertical     3   4   3   2   4   4   3    3    4    3
